# 00 · 분석할 데이터를 정한다

분석의 출발점은 어떤 기록을 남겼는지 밝히는 일이다. 이 노트북은 원본 JSON을 읽어 구간·검사 기록·산모 단위의 표와 신호 배열을 만들고, 제외한 이유를 함께 남긴다.

**처리 흐름:** JSON 3종 연결 → 라벨·신호 검증 → 결측 구간 제외 → 구간·기록·산모 표 생성 → 저장

**입력:** `input_data/`의 원본 JSON · **출력:** `data/prepared/`의 표, 신호 배열, 설정 정보

$$x_i\in\mathbb R^{2\times150},\qquad y_i\in\{0,1\}.$$

구간 $i$는 5분 길이다. 태아 심박수(FHR, bpm)와 자궁수축 신호(TOCO, 원본 척도)를 2초마다 측정하므로 채널마다 150개 값이 있다. $y_i=1$은 전문가가 해당 구간을 비정상으로 판독했다는 뜻이다. 출생 결과는 별도 변수로 보관한다.

이 노트북이 있는 폴더에서 실행한다. Python 3.10 이상과 NumPy, pandas, Jupyter가 필요하다. 아래에서 원본 경로와 결측 허용률을 정한다.

다른 컴퓨터로 옮길 때는 두 노트북과 `input_data/`를 복사한다. 기존 산출물도 쓰려면 `data/`를 함께 옮긴다. 산출물이 없으면 00, 01 순서로 실행한다. 데이터 폴더는 Git에서 제외되므로 저장소만 복사해서는 충분하지 않다.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

DATA = Path("input_data")
DATASET = Path("data/prepared")
MAX_MISSING_FRACTION = 0.30

### 출력에 남기는 오류 정보

오류는 고정된 분류와 건수로만 요약한다. `other_error`의 상세 원인은 내부 오류 파일에서 확인한다. 건수는 오류 내역 수이며, 같은 기록이 여러 번 포함될 수 있다.

In [ ]:
def issue_category(reason):
    # Match known phrases, but return ONLY fixed labels: never echo an exception.
    # Example: "FileNotFoundError: /private/DEMO.json" -> "missing_input".
    text = str(reason)
    groups = {
        "missing_input": ("FileNotFoundError", "No such file", "[Errno 2]"),
        "missing_dependency": ("No module named", "cannot import name", "ImportError"),
        "fhr_missing": ("FHR 결측률",),
        "invalid_wave": ("FHR과 TOCO", "측정 간격", "FHR은 0 이상", "신호 길이"),
        "invalid_bbox": ("Bbox",),
        "invalid_id": ("ID가", "산모 ID", "산모 목록", "행 정보 불일치", "구간 정보 불일치"),
        "invalid_labels": ("라벨이 올바르지", "라벨 수", "라벨·점수"),
        "no_usable_data": ("구간이 없습니다", "비었", "남은 산모가 없습니다"),
        "split_unavailable": ("분할", "양성·음성", "정상·비정상", "least populated class", "test_size"),
        "checkpoint_mismatch": ("체크포인트", "CNN 입력·설정"),
        "io_error": ("PermissionError", "Permission denied", "No space left", "OSError"),
        "invalid_value": ("ValueError", "TypeError", "유한", "could not convert", "ufunc"),
    }
    for category, phrases in groups.items():
        if any(phrase in text for phrase in phrases):
            return category
    return "other_error"

In [ ]:
def show_issue_counts(reasons):
    # Count issue entries, not unique people; one record may have several entries.
    categories = pd.Series([issue_category(reason) for reason in reasons], dtype="string")
    counts = categories.value_counts().rename_axis("category").to_frame("count")
    print("제외·오류 내역 수 =", len(categories))
    # Full messages, IDs, paths, and jobs stay in the internal CSV files.
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(counts)

## 1. 같은 기록끼리 연결한다

하나의 검사 기록은 세 파일이 맞아야 성립한다. `annotation_person`, `labels`, `emr`에서 같은 `{ID}.json`을 연결한다. 어느 폴더에든 있는 ID를 모두 수집해야 짝이 없는 기록도 제외 사유에 남는다.

단태아는 모든 구간을 사용한다. 쌍태아는 기록 ID의 끝자리 `_1` 또는 `_2`에 따라 `twin1` 또는 `twin2` 구간을 고른다. 선택한 구간은 원본 순서대로 `Abnormality`의 0·1 라벨에 대응시킨다. 구간 수와 라벨 수가 같고 산모 ID가 있어야 다음 단계로 넘어간다.

출력에는 폴더 종류, 처리할 기록 수, ID 5개만 보여준다. `record_ids` 전체를 출력하면 결과가 너무 길어진다. 뒤의 `read_record()`는 `paths[kind][record_id]`로 파일을 읽고, 전체 처리 반복문은 `record_ids`를 순회한다.


In [ ]:
kinds = ("annotation_person", "labels", "emr")

# 1. gather paths
# paths["annotation_person"][record_id]  # 해당 기록의 annotation JSON 경로
# paths["labels"]           [record_id]  # 해당 기록의 labels JSON 경로
# paths["emr"]              [record_id]  # 해당 기록의 emr JSON 경로

paths = {}
for kind in kinds:
    directory = DATA / kind
    if not directory.is_dir():
        raise FileNotFoundError(f"입력 폴더가 없습니다: {directory.resolve()}")

    files = {}
    for path in sorted(directory.glob("*.json")):
        files[path.stem] = path
    paths[kind] = files

# 2. gather record_ids
# record_id == 파일명 == (산모 ID)_(태아 구분 번호)

all_record_ids = set()
for files in paths.values():
    for record_id in files:
        all_record_ids.add(record_id)
record_ids = sorted(all_record_ids)

In [ ]:
# Report only directory names and aggregate file counts.
file_counts = pd.Series(
    {kind: len(files) for kind, files in paths.items()}, name="파일 수",
)
display(file_counts.to_frame())
print("paths keys =", list(paths))
print("record count =", len(record_ids))

산모 ID가 없으면 독립적인 평가 집단을 만들 수 없다. 공백과 결측 표시를 걸러 내되, 앞자리 0을 보존하도록 ID는 문자열로 다룬다.

In [ ]:
# 1. refine rough mother_id
def mother_id(value):
    value = str(value).strip()
    if value.lower() in (
        "", "9999", "9999.0", "none", "nan", "null", "<na>"
    ):
        raise ValueError("산모 ID가 없습니다")
    return value

# 2. visualize how refinery works
print(mother_id(" 00123 "))

try:
    mother_id("9999")
except ValueError as error:
    print('[details omitted from notebook output]')

파일명과 내부 ID가 일치해야 세 자료를 같은 기록으로 묶을 수 있다. 아래 함수는 한 기록을 네 단계로 처리한다.

1. 같은 ID의 JSON 세 파일을 읽는다.
2. ID와 산모·사이트·쌍태아 정보를 확인한다.
3. 단태아는 모든 구간, 쌍태아는 해당 태아의 구간만 고른다.
4. 선택한 구간과 라벨의 수·값을 확인하고 반환한다.

반환 순서는 `기본 정보, 선택한 구간, 0·1 라벨 배열, 원본 라벨 JSON, 원본 EMR JSON`이다. 읽기·검증에 실패한 기록은 뒤의 전체 처리 반복문에서 제외 사유로 남긴다.


In [ ]:
def read_record(record_id):
    
    # 1. convert JSON corresponding to record_id
    #    to 3 dictionaries
    #
    # extract annotation, label, emr

    documents = {}
    for kind in kinds: # ("annotation_person", "labels", "emr")
        path = paths[kind][record_id]
        documents[kind] = json.loads(path.read_text(encoding="utf-8-sig"))

    annotation = documents["annotation_person"]
    label = documents["labels"]
    emr = documents["emr"]

    for document in documents.values():
        if not isinstance(document, dict):
            raise ValueError("세 JSON 입력은 모두 객체여야 합니다")

    # ===========================================================

    # 2. extract base = {record_id, mother_id, site} 

    # check record_id consistency
    if str(label.get("ID")) != record_id or str(emr.get("ID")) != record_id:
        raise ValueError("파일명과 JSON의 ID가 다릅니다")

    # base = {record_id, mother_id, site}
    # mother id comes from emr
    base = dict(
        record_id=record_id,
        mother_id=mother_id(emr.get("Mother.de-identification_ID")),
        site=str(annotation.get("code") or "").split("_")[0].strip(),
    )

    # ===========================================================

    # 3. extract signal_parts from annotation['data']

    # check twin consistency
    twin = str(annotation.get("twins")).lower()
    if not base["site"] or twin not in ("true", "false"):
        raise ValueError("사이트 코드 또는 쌍태아 표기가 올바르지 않습니다")

    # Singleton: Select all segments.
    if twin == "false":
        signal_parts = list(annotation["data"])
    
    # Twins: Select only the segments for the corresponding fetus.
    else:
        # extract twin_number
        twin_number = record_id.rsplit("_", 1)[-1]
        if twin_number not in ("1", "2"):
            raise ValueError("쌍태아 기록 ID가 올바르지 않습니다")

        # Fictional example: annotation['data'][idx]['partial_image'] 
        #      = "demo_DEMO684219_20910417_twin2_p002.png"
        # 
        # question: does it contain 'twin2'?

        twin_token = f"twin{twin_number}"
        pattern = rf"(?:^|_){twin_token}(?:_|\.|$)"
        signal_parts = [
            part for part in annotation["data"]
            if re.search(pattern, part["partial_image"].lower())
        ]

    # =========================================================== 

    # 4. extract abnormal_labels from label
    # label["Abnormality"] = "1_1_0_0"

    # convert abnormality into list
    abnormal_labels = str(label["Abnormality"]).split("_")
    
    # check consistency between  (i) signal_parts
    #                           (ii) abnormal_labels
    if not signal_parts:
        raise ValueError("라벨이 올바르지 않거나 선택한 구간 수와 라벨 수가 다릅니다")
    if len(signal_parts) != len(abnormal_labels):
        raise ValueError("라벨이 올바르지 않거나 선택한 구간 수와 라벨 수가 다릅니다")
    if any(value not in ("0", "1") for value in abnormal_labels):
        raise ValueError("라벨이 올바르지 않거나 선택한 구간 수와 라벨 수가 다릅니다")

    abnormal_labels = np.asarray(abnormal_labels, dtype=int)
    
    # ===========================================================

    return base, signal_parts, abnormal_labels, label, emr

## 2. 라벨과 신호의 구조를 확인한다

제외 범위는 오류 종류에 따라 다르다. 기록 전체를 제외하면 그 기록의 모든 구간을 함께 제외한다.

| 조건 | 처리 |
| --- | --- |
| 파일 누락, ID·라벨 불일치, 산모 ID 누락 등 기록 검증 실패 | 기록 전체 제외 |
| Bbox 좌표·라벨 불일치 또는 신호 길이·값·측정 간격 오류 | 기록 전체 제외 |
| FHR 결측률이 `MAX_MISSING_FRACTION` 초과 | 해당 구간 제외 |
| 통과한 구간이 하나도 없음 | 기록 제외 |
| Bbox 없음 | 미검증 상태로 보존 |

### 2.1. Bbox 좌표와 라벨의 대응

비정상 표시 상자가 있다면, 그 위치도 라벨과 맞아야 한다. 이미지 너비가 $W$, 높이가 $H$, 구간 수가 $N$일 때, 0부터 세는 $k$번째 구간의 경계는 $(kW/N,0,(k+1)W/N,H)$다.

`Bbox`는 이 경계에서 좌표별로 0.5픽셀까지 벗어날 수 있다. 상자가 가리키는 구간은 비정상 라벨의 위치와 일치해야 한다. 이 검사는 JSON 좌표만 확인한다. 상자가 없으면 검증하지 않은 상태로 남기며, 실제 이미지 내용을 확인했다고 보지 않는다.


In [ ]:
def check_boxes(label, abnormal_labels):
    # Example:
    # abnormal_labels = [0, 1, 0, 0]
    # label["Bbox"] = [[100, 0, 200, 100, 400, 100]]
    # → Four segments; the Bbox marks segment 1.
    boxes = label.get("Bbox") or []
    if not boxes:
        return "not_provided"

    box_positions = []

    for box in boxes:
        # Example box: [x1, y1, x2, y2, image_width, image_height]
        #              [100, 0, 200, 100, 400, 100]
        values = np.asarray(box, dtype=float)
        x1, y1, x2, y2, image_width, image_height = values

        if (
            not np.isfinite(values).all()
            or image_width <= 0
            or image_height <= 0
        ):
            raise ValueError("Bbox 크기 또는 좌표가 올바르지 않습니다")

        # Example: 400 pixels / 4 labels = 100 pixels per segment.
        # The box starts at x=100, so its segment index is 1.
        segment_width = image_width / len(abnormal_labels)
        position = int(round(x1 / segment_width))

        # Example: segment 1 should cover x=100–200 and y=0–100.
        # Compare that expected box with the coordinates in the JSON.
        expected = np.array([
            position * segment_width, 0,
            (position + 1) * segment_width, image_height,
        ])
        actual = np.array([x1, y1, x2, y2])

        if np.max(np.abs(actual - expected)) > 0.5:
            raise ValueError("Bbox가 구간 경계와 맞지 않습니다")

        box_positions.append(position)

    # Example: box_positions = [1], abnormal_positions = [1].
    # Both must identify the same abnormal segments.
    abnormal_positions = np.flatnonzero(abnormal_labels).tolist()
    if sorted(box_positions) != abnormal_positions:
        raise ValueError("Bbox 위치와 비정상 라벨이 다릅니다")

    return "json_geometry_checked"

In [ ]:
# 구간 4개 중 1번 구간만 비정상
abnormal_labels = np.array([0, 1, 0, 0])

# Bbox: [x1, y1, x2, y2, 전체 이미지 너비, 전체 이미지 높이]
label = {
    "Bbox": [[100, 0, 200, 100, 400, 100]]
}

result = check_boxes(label, abnormal_labels)
print(result)  # json_geometry_checked

### 2.2. 신호 구조와 결측률 기준

신호의 구조와 결측률은 서로 다른 기준으로 판단한다. 두 채널에는 각각 유한한 값 150개가 있어야 하며, FHR은 음수일 수 없다. FHR의 0은 결측으로 센다.

$$q_i=\frac1{150}\sum_{t=1}^{150}\mathbf1[F_{it}=0],\qquad
q_i>0.30\ \Longrightarrow\ \text{구간 제외}.$$

$F_{it}$는 구간 $i$의 $t$번째 FHR 값이며, $\mathbf1[A]$는 조건 $A$가 참일 때 1이다. 위 식은 기본 설정인 30%를 나타낸다. 실제 기준은 첫 코드 셀의 `MAX_MISSING_FRACTION`이며, 기준과 같으면 남긴다.

길이·측정 간격·라벨 대응에 문제가 있으면 검사 기록 전체를 제외한다. 결측률만 기준을 넘으면 해당 구간만 제외한다. 남긴 신호에는 평활화나 보간을 적용하지 않는다.

In [ ]:
def read_wave(part):
    # Fictional example: part["fhr"] = "153,149,151,..."
    #                    part["toco"] = "18,21,17,..."
    # Each channel may also already be a list of values.
    fhr_values = part["fhr"]
    toco_values = part["toco"]

    # Split comma-separated strings into individual measurements.
    if isinstance(fhr_values, str):
        fhr_values = fhr_values.split(",")
    if isinstance(toco_values, str):
        toco_values = toco_values.split(",")

    # Row 0 is FHR; row 1 is TOCO.
    # Each row must contain 150 measurements.
    wave = np.asarray([fhr_values, toco_values], dtype=float)
    if wave.shape != (2, 150):
        raise ValueError("FHR과 TOCO에는 각각 값 150개가 필요합니다")

    if not np.isfinite(wave).all():
        raise ValueError("FHR과 TOCO의 값은 모두 유한해야 합니다")

    # Measurements are 2 seconds apart, and FHR cannot be negative.
    if float(part["interval_sec"]) != 2:
        raise ValueError("측정 간격은 2초여야 합니다")
    if (wave[0] < 0).any():
        raise ValueError("FHR은 0 이상이어야 합니다")

    return wave

### 2.3. 구간별 결측 기준 적용

결측 기준은 구간마다 적용한다. 기준을 넘는 구간은 제외 내역에 넣고, 통과한 구간에는 원래 순번과 파일명을 남긴다. 따라서 일부 구간을 제외해도 원본에서의 위치를 찾을 수 있다.

In [ ]:
def select_segments(record_id, signal_parts, abnormal_labels):
    rows = []
    waves = []
    dropped = []

    # Record-level flag:
    #   [0, 0, 0] -> 1  (all segments are normal)
    #   [0, 1, 0] -> 0  (at least one segment is abnormal)
    record_all_normal = int(not abnormal_labels.any())

    for segment_index, (part, target) in enumerate(
        zip(signal_parts, abnormal_labels)
    ):
        wave = read_wave(part)

        # FHR values of 0 count as missing.
        # Example: 30 zeros among 150 values → missing_fraction = 0.20.
        missing_fraction = float(np.mean(wave[0] == 0))

        # Drop the segment only if its missing fraction exceeds the limit.
        # A segment exactly at the limit is kept.
        if missing_fraction > MAX_MISSING_FRACTION:
            dropped.append(
                dict(
                    record_id=record_id,
                    seg_idx=segment_index,
                    scope="window",
                    reason="FHR 결측률이 기준을 초과함",
                )
            )
            continue

        # Keep the segment's metadata and its wave at the same list position.
        rows.append(
            dict(
                row_id=f"{record_id}:{segment_index}",
                seg_idx=segment_index,
                target=int(target),
                missing_fraction=missing_fraction,
                source_name=part["partial_image"],
                record_all_normal=record_all_normal,
            )
        )
        waves.append(wave)

    return rows, waves, dropped

## 3. 기록별 요약과 임상값을 계산한다

### 3.1. 원래 구간을 기준으로 판독 요약

기록의 판독 요약은 원래 선택한 태아의 모든 구간을 기준으로 계산한다. 결측 때문에 제외한 구간의 라벨도 포함한다. 비정상 비율은 $a_r=N_r^{-1}\sum_j y_{rj}$이며, $N_r$는 해당 태아의 원래 구간 수다.

최장 연속 비정상 구간 수 $\ell_r$는 파일명으로 계산한다. 앞부분이 같고 `_p` 뒤 번호가 하나씩 늘어날 때만 연속으로 센다. 파일명을 해석할 수 없으면 결측으로 남긴다. 파일명 순서는 연속성의 단서일 뿐, 실제 시간 간격을 입증하지는 않는다.

In [ ]:
def longest_run(abnormal_labels, names):
    previous, current, longest = None, 0, 0
    for flag, name in zip(abnormal_labels, names):
        match = re.fullmatch(
            r"(.+)_p(\d+)(?:\.png)?",
            name,
            re.IGNORECASE,
        )
        if match is None:
            return np.nan
        item = (match[1], int(match[2]))
        # 파일 번호가 끊기면 연속 구간도 끊긴다.
        adjacent = (
            previous is not None
            and item[0] == previous[0]
            and item[1] == previous[1] + 1
        )
        if not flag:
            current = 0
        elif adjacent:
            current += 1
        else:
            current = 1
        longest = max(longest, current)
        previous = item
    return longest

### 3.2. 임상값과 출생 결과 변환

임상값은 원본의 의미를 살려 옮긴다. 산모 관련 11개 항목에 산모 나이와 임신 주수를 더한다. 숫자로 바꿀 수 없는 값, 무한대, 결측 표시인 9999는 모두 결측으로 처리한다.

원본 날짜는 `YYYYMM` 형식이다. 산모 나이는 출생 연도에서 산모의 출생 연도를 뺀 뒤 1을 더한다. 따라서 만 나이로 해석하면 안 된다. 임신 주수는 주 수에 일 수/7을 더하며, 일 수 항목 자체가 없을 때만 0일로 둔다.

제대동맥 pH, 아프가 점수, 전문가 판독 항목은 각각 별도 열로 보관한다. 구간의 비정상 라벨과 출생 결과를 구별하기 위해서다.

In [ ]:
clinical_fields = dict(
    height="Height",
    weight="Weight",
    gravida="Gravida",
    para="Para",
    sbp="SBP",
    dbp="DBP",
    ghtn="GHTN",
    hypertension="Hypertension",
    gdm="GDM",
    dm="DM",
    preeclampsia="pre-eclampsia",
)
label_fields = (
    "BaseLine",
    "Baseline_Variability",
    "Acceleration",
    "Early_deceleration",
    "Late_deceleration",
    "Variable_deceleration",
    "Prolonged_deceleration",
)
outcome_fields = dict(
    ua_ph="UA.pH",
    apgar1="APGAR.1min",
    apgar5="APGAR.5min",
)

알 수 없는 임상값은 결측으로 남긴다. 아래 함수는 숫자 변환과 결측 표시 처리를 통일하고, 같은 규칙으로 나이·임신 주수·출생 결과를 계산한다.

In [ ]:
def numeric(value):
    # Example: "155.0" -> 155.0
    # Missing or invalid values, including 9999, become np.nan.
    try:
        number = float(value)
    except (TypeError, ValueError):
        return np.nan

    if not np.isfinite(number) or number == 9999:
        return np.nan

    return number

In [ ]:
def clinical_values(emr):
    values = {}

    # Example: "height" reads emr["Mother.Height"].
    for key, field in clinical_fields.items():
        values[key] = numeric(emr.get(f"Mother.{field}"))

    # Fictional dates use YYYYMM: 209104 // 100 = 2091.
    # Fictional age calculation: 2091 - 2062 + 1 = 30.
    birth_year = numeric(emr.get("Birth Date")) // 100
    mother_birth_year = numeric(emr.get("Mother.Birth Date")) // 100
    values["maternal_age"] = birth_year - mother_birth_year + 1

    # Example: 40 weeks and 3 days -> 40 + 3/7 weeks.
    weeks = numeric(emr.get("GA.wks"))
    days = numeric(emr.get("GA.day", 0))
    values["gestational_age"] = weeks + days / 7

    # Example: "ua_ph" reads emr["UA.pH"].
    for key, field in outcome_fields.items():
        values[key] = numeric(emr.get(field))

    return values

## 4. 전체 기록을 처리한다

검사 기록은 모든 검사를 통과한 뒤 한 번에 추가한다. 마지막 구간에서 구조 오류가 나면 앞서 읽은 구간도 함께 제외해야 한다. 아래 반복문은 임시로 모은 구간과 신호를 검증이 끝난 뒤에만 최종 목록에 넣고, 실패한 기록에는 이유를 남긴다.

In [ ]:
# Collect record summaries, retained segments and waves, and exclusion reasons.
records = []
segments = []
waves = []
exclusions = []

for record_id in record_ids:
    try:
        # Read one record and validate its labels, Bboxes, and signal segments.
        base, signal_parts, abnormal_labels, label, emr = read_record(record_id)
        bbox_status = check_boxes(label, abnormal_labels)
        selected_rows, selected_waves, dropped_segments = select_segments(
            record_id, signal_parts, abnormal_labels
        )

        # Example: 3 labeled segments with 1 dropped -> n_segments=3, n_kept=2.
        segment_names = [part["partial_image"] for part in signal_parts]
        record_row = dict(
            **base,
            row_id=record_id,
            n_segments=len(abnormal_labels),
            n_kept=len(selected_rows),
            any_abnormal=int(abnormal_labels.any()),
            abnormal_fraction=float(abnormal_labels.mean()),
            longest_abnormal_run=longest_run(abnormal_labels, segment_names),
            bbox_status=bbox_status,
        )

        # Add numeric clinical values and label fields to the record summary.
        record_row.update(clinical_values(emr))
        for field in label_fields:
            record_row[f"label_{field}"] = numeric(label.get(field))

        # Keep window exclusions even when no segment survives the threshold.
        exclusions.extend(dropped_segments)
        if not selected_rows:
            raise ValueError("조건을 충족하는 신호 구간이 없습니다")

        # Build all segment rows before adding this record to the final lists.
        record_segments = [dict(**base, **row) for row in selected_rows]
        records.append(record_row)
        segments.extend(record_segments)
        waves.extend(selected_waves)

    except (
        KeyError,
        ValueError,
        TypeError,
        IndexError,
        AttributeError,
        OSError,
        OverflowError,
    ) as error:
        # seg_idx=-1 marks an exclusion of the whole record.
        exclusions.append(
            dict(
                record_id=record_id,
                seg_idx=-1,
                scope="record",
                reason=f"{type(error).__name__}: {error}",
            )
        )

print("남긴 검사 기록 수:", len(records))
print("남긴 구간 수:", len(segments))
print("제외 내역 수:", len(exclusions))

## 5. 구간·기록·산모 표를 만든다

### 5.1. 구간·기록 표와 신호 배열

최종 표와 신호 배열은 같은 구간 순서를 공유한다. 남은 구간에 0부터 시작하는 `signal_index`를 붙이고, 같은 순서로 신호를 쌓는다. 원래 구간 번호인 `seg_idx`는 따로 보존한다. 제외 내역은 단위와 사유별로 집계한다.

In [ ]:
# One row in segment_table describes one retained signal segment.
# Example: if an earlier segment was dropped, seg_idx=2 may have signal_index=1.
segment_columns = [
    "record_id",
    "mother_id",
    "site",
    "row_id",
    "seg_idx",
    "target",
    "missing_fraction",
    "source_name",
    "record_all_normal",
]
segment_table = pd.DataFrame(segments, columns=segment_columns)
segment_table["signal_index"] = np.arange(len(segment_table))

In [ ]:
# One row in record_table summarizes one retained recording.
label_columns = [f"label_{key}" for key in label_fields]
record_columns = [
    "record_id",
    "mother_id",
    "site",
    "row_id",
    "n_segments",
    "n_kept",
    "any_abnormal",
    "abnormal_fraction",
    "longest_abnormal_run",
    "bbox_status",
    *clinical_fields,
    "maternal_age",
    "gestational_age",
    "ua_ph",
    "apgar1",
    "apgar5",
    *label_columns,
]
record_table = pd.DataFrame(records, columns=record_columns)

In [ ]:
# Keep each excluded record or segment together with its reason.
exclusion_columns = ["record_id", "seg_idx", "scope", "reason"]
exclusion_table = pd.DataFrame(exclusions, columns=exclusion_columns)

In [ ]:
# signal_array[i] is the FHR/TOCO wave for segment_table row i.
if waves:
    signal_array = np.stack(waves)
else:
    signal_array = np.empty((0, 2, 150), dtype=float)

In [ ]:
# Fixed categories prevent IDs, paths, or offending values from reaching the output.
# Example: 3 missing-file records + 2 missing-FHR windows -> two category rows.
exclusion_groups = pd.DataFrame({
    "scope": exclusion_table["scope"],
    "category": exclusion_table["reason"].map(issue_category),
})
exclusion_summary = exclusion_groups.groupby(["scope", "category"]).size().to_frame("count")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(exclusion_summary)


### 5.2. 산모별 집계

산모 수는 검사 기록 수와 다르다. 같은 산모의 반복 검사와 쌍태아 기록을 `Mother.de-identification_ID`로 묶어 `mothers.csv`를 만든다. 각 산모에 대해 남은 기록 수, 구간 수, 비정상 구간의 유무를 센다. 남은 구간이 없는 산모는 이후 분할에 포함하지 않는다.

In [ ]:
# Group all retained segments by mother, including twins and repeat recordings.
# Fictional twins: "DEMO684219_1" and "DEMO684219_2" share mother "DEMO684219".
mother_table = segment_table.groupby("mother_id", as_index=False).agg(
    n_records=("record_id", "nunique"),
    n_windows=("row_id", "size"),
    any_abnormal=("target", "max"),
)

# any_abnormal is 1 if at least one retained segment has target=1.
mother_counts = pd.Series(
    {
        "산모 수": len(mother_table),
        "비정상 구간이 있는 산모": int(mother_table.any_abnormal.sum()),
    }
)
display(mother_counts.to_frame("수"))

### 5.3. 구간 표와 신호 배열의 연결 확인

아래는 남긴 구간 중 처음 3개다. `seg_idx`는 선택한 태아 구간의 원래 순번이고, `signal_index`는 저장할 신호 배열의 행 번호다. 중간 구간이 제외되면 두 값이 달라질 수 있다. 표의 각 행에 해당하는 신호는 `signal_array[signal_index]`로 찾는다.

배열의 축은 `(남긴 구간 수, 채널 수, 채널별 측정값 수)`이며, 채널 순서는 FHR, TOCO다.


In [ ]:
# Array axes: retained segment, channel (FHR then TOCO), measurement.
print("신호 배열 크기 (구간, 채널, 측정값):", signal_array.shape)

## 6. 전처리 결과는 한곳에 저장한다

`data/prepared/`에는 현재 설정으로 만든 데이터 한 벌만 둔다. 표는 구간의 뜻을 설명하고, 배열은 실제 신호를 담는다.

| 파일 | 내용 |
| --- | --- |
| `segments.csv` | 구간별 ID, 산모, 사이트, 라벨, 결측률, 신호 인덱스 |
| `mothers.csv` | 산모별 기록 수, 구간 수, 비정상 구간 유무 |
| `records.csv` | 검사 기록별 판독 요약, 임상값, 출생 결과 |
| `exclusions.csv` | 제외한 기록·구간과 그 이유 |
| `raw_signals.npy` | 구간 $n$개의 신호 배열, 크기 $(n,2,150)$ |

`row_id`는 기록 ID와 선택한 구간의 원래 순번을 합친 식별자다. `signal_index`는 제외 처리가 끝난 뒤 배열에서의 행 번호다. 둘을 구별해야 표와 신호를 올바르게 연결할 수 있다.

재실행하면 같은 파일을 갱신한다. 전처리 기준을 바꿨다면 00을 실행한 뒤 01과 후속 분석도 다시 실행한다. 보존할 실험 결과는 갱신 전에 별도로 옮겨 둔다.

In [ ]:
# Save the prepared tables together in one directory.
DATASET.mkdir(parents=True, exist_ok=True)

tables = {
    "segments": segment_table,
    "records": record_table,
    "mothers": mother_table,
    "exclusions": exclusion_table,
}
# Example: tables["segments"] -> DATASET / "segments.csv".
for name, table in tables.items():
    table.to_csv(DATASET / f"{name}.csv", index=False)

# Array row i matches segment_table row i and its signal_index=i.
np.save(DATASET / "raw_signals.npy", signal_array)

설정값은 저장된 데이터와 함께 남겨야 해석할 수 있다. `dataset.json`에 실제 적용한 결측 허용률과 원본·잔존 데이터 수를 기록한다. 다음 노트북은 이 값을 읽어 요약한다.

In [ ]:
# Store the preprocessing limit and output sizes with the saved data.
# Example: metadata["n_segments"] equals the number of rows in segments.csv.
metadata = dict(
    max_missing_fraction=MAX_MISSING_FRACTION,
    n_input_records=len(record_ids),
    n_records=len(record_table),
    n_segments=len(segment_table),
    n_mothers=segment_table.mother_id.nunique(),
)
metadata_path = DATASET / "dataset.json"
metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)
print("저장 위치:", DATASET)

## 7. 남긴 데이터와 제외한 데이터를 함께 센다

아래 요약은 원본 중 얼마나 남았는지 보여준다. 제외 기록 수와 제외 구간 수는 단위가 다르다. 모든 구간을 잃은 기록은 기록 제외에도 잡히므로 두 수를 더하면 중복 집계가 된다.

남은 데이터는 다음 노트북에서 산모 단위로 학습·검증·시험 집합에 배정한다.

In [ ]:
# A record with all segments dropped can appear in both exclusion counts.
# Count excluded records once, but count excluded windows individually.
excluded_records = exclusion_table.loc[
    exclusion_table.scope.eq("record"),
    "record_id",
].nunique()
excluded_segments = int(exclusion_table.scope.eq("window").sum())

if len(segment_table):
    status = "남은 데이터를 산모 단위로 분할할 준비가 됐습니다."
else:
    status = "조건을 충족하는 구간이 없습니다. 제외 내역에서 이유를 확인하십시오."

summary_text = (
    f"전체 기록 {len(record_ids):,}개 중 **{len(record_table):,}개를 남겼습니다**. "
    f"**산모 {segment_table.mother_id.nunique():,}명의 구간 {len(segment_table):,}개**입니다. "
    f"제외한 기록은 {excluded_records:,}개이며, 결측률 때문에 제외한 구간은 "
    f"{excluded_segments:,}개입니다.\n\n{status}"
)
display(Markdown(summary_text))